# Phase 18 — Production Monitoring

**Monthly monitoring dashboard** showing:
- Current portfolio weights and momentum scores
- Performance vs SPY (MTD, YTD, 1Y, 3Y, ITD)
- Rolling Sharpe and drawdown
- Trade sheet: what to buy/sell this month

Run this every month-end to track the strategy.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings('ignore')

from src.data import load_prices, compute_returns
from src.signals import load_signals
from src.monitoring import run_monitoring, rolling_monitor

plt.rcParams.update({'figure.dpi':130,'font.size':10,'axes.titlesize':11,
                     'axes.spines.top':False,'axes.spines.right':False})
print('Ready.')

In [ ]:
prices  = load_prices(directory='../data/processed')
returns = compute_returns(prices)
signals = load_signals('../data/processed')

res = run_monitoring(signals, prices, returns, proc_dir='../data/processed')
port      = res['portfolio']
perf      = res['performance']
roll_df   = res['rolling']
hrp_ret   = res['hrp_ret']
spy_ret   = res['spy_ret']
rf_mo     = res['rf_monthly']

## Performance Dashboard

In [ ]:
periods  = list(perf.keys())
hrp_cg   = [perf[p]['HRP'].get('CAGR %', float('nan')) for p in periods]
spy_cg   = [perf[p]['SPY'].get('CAGR %', float('nan')) for p in periods]
hrp_sh   = [perf[p]['HRP'].get('Sharpe', float('nan')) for p in periods]
hrp_dd   = [perf[p]['HRP'].get('MaxDD %', float('nan')) for p in periods]

valid    = [(i,p) for i,p in enumerate(periods)
            if not (pd.isna(hrp_cg[i]) or pd.isna(spy_cg[i]))]
vi, vp   = zip(*valid) if valid else ([], [])
vi = list(vi); vp = list(vp)
h_cg_v   = [hrp_cg[i] for i in vi]
s_cg_v   = [spy_cg[i] for i in vi]

fig, axes = plt.subplots(2, 2, figsize=(15, 9))

ax = axes[0, 0]
x  = range(len(vi))
w  = 0.38
b1 = ax.bar([i-w/2 for i in x], h_cg_v, w, label='HRP Strategy', color='#2196F3', alpha=0.85)
b2 = ax.bar([i+w/2 for i in x], s_cg_v, w, label='SPY B&H',      color='#FF9800', alpha=0.85)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(list(x)); ax.set_xticklabels(vp)
ax.set_ylabel('CAGR (%)'); ax.set_title('CAGR by Period: HRP vs SPY', fontweight='bold')
ax.legend(fontsize=9)
for bar in list(b1)+list(b2):
    h = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2, h+0.2 if h>=0 else h-0.8,
            f'{h:.1f}%', ha='center', fontsize=7.5)

ax = axes[0, 1]
wts  = port['weights']
lbls = [f'{k}\n{v*100:.1f}%' for k,v in wts.items()]
ax.pie(wts.values, labels=lbls, autopct='%1.0f%%',
       colors=plt.cm.Set2.colors[:len(wts)],
       wedgeprops={'edgecolor':'white','linewidth':1.5})
ax.set_title(f'Current Portfolio ({port["date"].strftime("%b %Y")})', fontweight='bold')

ax = axes[1, 0]
ax.plot(roll_df.index, roll_df['Rolling Sharpe'], color='#9C27B0', lw=2)
ax.axhline(1.0, color='#4CAF50', lw=1, ls='--', label='Sharpe=1.0')
ax.axhline(0.7, color='#FF9800', lw=1, ls='--', label='Sharpe=0.7')
ax.axhline(0,   color='black',   lw=0.8)
ax.fill_between(roll_df.index, roll_df['Rolling Sharpe'], 0,
                where=roll_df['Rolling Sharpe']>0, alpha=0.15, color='#9C27B0')
ax.set_ylabel('Rolling 12m Sharpe'); ax.set_title('Rolling Sharpe Ratio', fontweight='bold')
ax.legend(fontsize=9)

ax = axes[1, 1]
eq = (1 + hrp_ret).cumprod()
dd = (eq / eq.cummax() - 1) * 100
ax.fill_between(dd.index, dd.values, 0, alpha=0.4, color='#F44336')
ax.plot(dd.index, dd.values, color='#F44336', lw=1)
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Drawdown (%)'); ax.set_title('Drawdown from Peak', fontweight='bold')
curr_dd = float(dd.iloc[-1])
ax.text(dd.index[-1], curr_dd-1, f'Now: {curr_dd:.1f}%', ha='right', fontsize=9,
        color='#F44336', fontweight='bold')

plt.suptitle(f'Monthly Monitoring Dashboard — {port["date"].strftime("%B %Y")}',
             fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0,0,1,0.96]); plt.show()

## Trade Sheet — What to Do This Month

In [ ]:
print('=' * 55)
print(f'  TRADE SHEET — {port["date"].strftime("%B %Y")}')
print('=' * 55)
print(f'  {"ETF":<6}  {"Curr Wt":>8}  {"Prev Wt":>8}  {"Change":>9}  Action')
print(f'  {"-"*6}  {"-"*8}  {"-"*8}  {"-"*9}  {"-"*8}')

all_tkrs = sorted(set(list(port['weights'].index)) | set(list(port['prev_weights'].index)))
for tkr in all_tkrs:
    curr = port['weights'].get(tkr, 0.0)
    prev = port['prev_weights'].get(tkr, 0.0)
    chg  = curr - prev
    if curr > 0 and prev == 0:
        action = 'BUY (new)'
    elif curr == 0 and prev > 0:
        action = 'SELL (exit)'
    elif abs(chg) > 0.005:
        action = 'INCREASE' if chg > 0 else 'REDUCE'
    else:
        action = 'hold'
    print(f'  {tkr:<6}  {curr*100:>7.1f}%  {prev*100:>7.1f}%  {chg*100:>+8.1f}%  {action}')

trade_mag = port['trade_sheet'].abs().sum()
print(f'  {"-"*55}')
print(f'  Total turnover this month : {trade_mag*100:.1f}% (one-way)')
print('=' * 55)

## Rolling Active Return vs SPY

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

ax1.plot(roll_df.index, roll_df['Rolling CAGR %'], label='HRP Rolling CAGR%', color='#2196F3', lw=2)
spy_roll = spy_ret.reindex(hrp_ret.index).fillna(0)
spy_eq = (1 + spy_roll).cumprod()

ax1.plot(roll_df.index, roll_df['Active CAGR %'], label='Active CAGR% (HRP-SPY)', color='#4CAF50', lw=1.5, ls='--')
ax1.axhline(0, color='black', lw=0.8)
ax1.set_ylabel('Rolling 12m CAGR (%)')
ax1.set_title('Rolling 12m Performance', fontweight='bold')
ax1.legend(fontsize=9)

eq   = (1 + hrp_ret).cumprod()
spy_eq_full = (1 + spy_roll).cumprod()
ax2.plot(eq.index, eq.values,         label='HRP Strategy', color='#2196F3', lw=2)
ax2.plot(spy_eq_full.index, spy_eq_full.values, label='SPY B&H',    color='#FF9800', lw=1.5, ls='--')
ax2.set_ylabel('Growth of £1')
ax2.set_title('Cumulative Equity', fontweight='bold')
ax2.legend(fontsize=9)

plt.suptitle('Strategy Performance Over Time', fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0,0,1,0.96]); plt.show()

last_roll = roll_df.iloc[-1]
print(f'Latest rolling (12m) Sharpe : {last_roll["Rolling Sharpe"]:.3f}')
print(f'Latest rolling (12m) CAGR   : {last_roll["Rolling CAGR %"]:.2f}%')
print(f'Latest rolling active return : {last_roll["Active CAGR %"]:.2f}%')